# s02 — The `calculate` Tool

**What this teaches:** how to bolt a *single* deterministic tool onto the loop you saw in [s01_loop](../s01_loop/s01_loop.ipynb). The model is unreliable at multi-step arithmetic, so we offload each expression to `govaluate` via the `calculate` tool and get exact results back.

**Concept anchor:** tools are the agent's hands. A tool turns a fuzzy LLM into a deterministic actor for a narrow task — here, arithmetic. The loop dispatches each tool call, feeds the result back, and the model continues until it has no more calls to make.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars before launching Jupyter, e.g.:
  ```
  export OMNIS_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export OMNIS_MODEL=qwen2.5-coder
  ```
  Defaults to a local Ollama. Swap in `anthropic` / `openai` / `gemini` for hosted providers — see [docs/providers.md](../../docs/providers.md).

## 1. Imports

We add `core/tools` (aliased `fstools`) on top of the s01 imports — that's the package that exposes `NewCalcTools()`.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
    fstools "github.com/blouargant/omnis/core/tools"
)

## 2. Helper

Same panic-based `must` we used in s01 — survives a notebook error without taking down the kernel.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Construct the model

Identical to s01: `agentkit.NewModel` reads the provider/key/model from env or `config/agent.yaml`.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)

## 4. Build the agent with the calculator tool

Two new things versus s01: an `Instruction` that *forces* the model to delegate every arithmetic step to the tool, and `Tools: fstools.NewCalcTools()` — the slice of `*tool.FunctionTool` instances ADK will register on the model side.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s02_calc",
    Description: "Calculator tool demo.",
    Model:       llm,
    Instruction: "Use the calculate tool for every arithmetic step. " +
        "Never do the math in your head. Show each call's result inline.",
    Tools: fstools.NewCalcTools(),
})
must(err)

## 5. Build the runner and run

Watch the stream: the model emits a `calculate` tool call, gets the exact numeric result, then chains the next expression. You'll typically see two or three tool round-trips before the final answer.

In [ ]:
r, err := agentkit.Runner("s02", a)
must(err)

prompt := "Compute (sqrt(2) + 3*4) ** 2 and then divide that by 7. Show your work."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- Multiple `calculate` tool-call events inside a single `RunOnce` — the loop continues until the model emits a plain text answer.
- The numbers are **exact** (no floating-point hallucination). Compare with [s01_loop](../s01_loop/s01_loop.ipynb), where the model has to invent answers in prose.
- For a larger tool surface (filesystem operations), jump to [s05_tools](../s05_tools/s05_tools.ipynb).

## Try it yourself

1. Change the prompt to ask for compound interest over 30 years — verify the model calls `calculate` for each year.
2. Drop the `Instruction` and re-run: the model may try to do the math in its head and produce a wrong answer.